## Data Sources Merge and Clean-up

#### Data Scope between 2008-2012

### Importing packages

In [ ]:
install.packages("readr")
install.packages("dplyr")
install.packages("tidyr")
install.packages("stringr")
install.packages("httr")
install.packages("jsonlite")

In [288]:
library(readr)
library(dplyr)
library(tidyr)
library(stringr)
library(httr)
library(jsonlite)

### Reading the main Wisconsin Data

In [289]:
wis <- read.csv("main_merged_data_FIPS.csv",, header = TRUE) #read main Wisconsin data

### A. USDA Education data merge and cleanup with main Wisconsin Data

In [290]:
edu <- read.csv("Education1970-2023.csv", header = TRUE)

##### Pivoting data to merge with Main Wisconsin data

In [291]:
#Ensure only Wisconsin's FIPS codes are present
edu <- edu %>%
  filter(FIPS.Code >= 55001 & FIPS.Code <= 55141) 
head(edu,10)

,FIPS.Code,State,Area.name,Attribute,Value
,<int>,<chr>,<chr>,<chr>,<dbl>
1,55001,WI,Adams County,2013 Urban Influence Code,6.0
2,55001,WI,Adams County,2024 Urban Influence Code,6.0
3,55001,WI,Adams County,2013 Rural-urban Continuum Code,8.0
4,55001,WI,Adams County,2023 Rural-urban Continuum Code,8.0
5,55001,WI,Adams County,"Less than a high school diploma, 1970",3342.0
6,55001,WI,Adams County,"High school diploma only, 1970",1494.0
7,55001,WI,Adams County,"Some college (1-3 years), 1970",329.0
8,55001,WI,Adams County,"Four years of college or higher, 1970",259.0
9,55001,WI,Adams County,"Percent of adults with less than a high school diploma, 1970",61.6


In [292]:
# Pivot wider: Attribute -> columns, Value -> values
edu_wide <- edu %>%
  pivot_wider(
    id_cols = c(FIPS.Code, State, Area.name),  
    names_from = Attribute,                       
    values_from = Value                           
  )

In [293]:
#Limiting my analysis to 2000-2018 initially. Removing columns corresponding to any other year
edu_wide <- edu_wide %>%
  select(-matches("1970|1980|1990|2019|2024|2023"))

In [294]:
# Renaming FIPS column

edu_wide <- edu_wide %>%
  rename(FIPS = FIPS.Code)
head(edu_wide)

FIPS,State,Area.name,2013 Urban Influence Code,2013 Rural-urban Continuum Code,"Less than high school graduate, 2000","High school graduate (or equivalency), 2000","Some college or associate degree, 2000","Bachelor's degree or higher, 2000","Percent of adults who are not high school graduates, 2000",⋯,"Percent of adults completing some college or associate degree, 2000","Percent of adults with a bachelor's degree or higher, 2000","Less than high school graduate, 2008-12","High school graduate (or equivalency), 2008-12","Some college or associate degree, 2008-12","Bachelor's degree or higher, 2008-12","Percent of adults who are not high school graduates, 2008-12","Percent of adults who are high school graduates (or equivalent), 2008-12","Percent of adults completing some college or associate degree, 2008-12","Percent of adults with a bachelor's degree or higher, 2008-12"
<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
55001,WI,Adams County,6,8,3202,5755,3404,1369,23.3,⋯,24.8,10.0,2413,6863,5075,1922,14.828243,42.17415,31.18663,11.81098
55003,WI,Ashland County,11,7,1697,4322,2888,1761,15.9,⋯,27.1,16.5,982,3754,3587,2506,9.068243,34.66617,33.12402,23.14156
55005,WI,Barron County,6,6,5281,11816,8392,4453,17.6,⋯,28.0,14.9,3948,13082,10106,5172,12.219884,40.49152,31.28018,16.00842
55007,WI,Bayfield County,6,8,1374,3598,3283,2271,13.1,⋯,31.2,21.6,790,3732,3861,3064,6.901372,32.60243,33.72936,26.76684
55009,WI,Brown County,2,2,19743,50311,41729,32389,13.7,⋯,28.9,22.5,15886,53689,50735,41925,9.791968,33.09335,31.27254,25.84214
55011,WI,Buffalo County,6,8,1488,4110,2468,1318,15.9,⋯,26.3,14.0,1029,4170,2732,1670,10.717634,43.43298,28.45537,17.39402


#### Retaining only columns that have percentage data as it makes the most sense.

In [295]:
# Geographical columns
core_cols <- c("FIPS", "State", "Area.name")

# Urban/Rural classification columns
urban_rural_cols <- c(
  "2013 Urban Influence Code",
  "2013 Rural-urban Continuum Code"
)

#  Columns that contain 'Percent' 
percent_cols <- names(edu_wide)[
  grepl("Percent", names(edu_wide)) 
]

# Combine all and create a new dataframe
edu_selected <- edu_wide %>%
  select(all_of(core_cols), all_of(urban_rural_cols), all_of(percent_cols))


In [296]:
head(edu_selected)

FIPS,State,Area.name,2013 Urban Influence Code,2013 Rural-urban Continuum Code,"Percent of adults who are not high school graduates, 2000","Percent of adults who are high school graduates (or equivalent), 2000","Percent of adults completing some college or associate degree, 2000","Percent of adults with a bachelor's degree or higher, 2000","Percent of adults who are not high school graduates, 2008-12","Percent of adults who are high school graduates (or equivalent), 2008-12","Percent of adults completing some college or associate degree, 2008-12","Percent of adults with a bachelor's degree or higher, 2008-12"
<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
55001,WI,Adams County,6,8,23.3,41.9,24.8,10.0,14.828243,42.17415,31.18663,11.81098
55003,WI,Ashland County,11,7,15.9,40.5,27.1,16.5,9.068243,34.66617,33.12402,23.14156
55005,WI,Barron County,6,6,17.6,39.5,28.0,14.9,12.219884,40.49152,31.28018,16.00842
55007,WI,Bayfield County,6,8,13.1,34.2,31.2,21.6,6.901372,32.60243,33.72936,26.76684
55009,WI,Brown County,2,2,13.7,34.9,28.9,22.5,9.791968,33.09335,31.27254,25.84214
55011,WI,Buffalo County,6,8,15.9,43.8,26.3,14.0,10.717634,43.43298,28.45537,17.39402


In [297]:
#Longer format of the data to merge with main Wisconsin data by year

edu_long <- edu_selected %>%
  pivot_longer(
    cols = starts_with("Percent of adults"),   # columns to reshape
    names_to = c("Measure", "Year"),          # split into Measure and Year
    names_pattern = "(.*), (\\d{4}-?\\d{0,2})", # regex to separate measure and year
    values_to = "Percent"                      # new column for values
  ) %>%
  mutate(
    Year = ifelse(Year == "", NA, Year)       # handle any missing years
  ) %>%
  select(FIPS, State, Area.name, `2013 Urban Influence Code`,
         `2013 Rural-urban Continuum Code`, Year, Measure, Percent)

#### Data Imputation
- The years in my main wisconsin data are all individual years and my education data from USDA is all decade-wise. 
- USDA has either 2000 or 2008-12(comprising of 2008-2012 a 5 year ACS data)
- Since there is no specific data for the years 2001-2007, I will be using proxy data that will represent these years as follows:
  - For years 2001-2004, I will use the year 2000 data as proxy.(representative of early 2000s)
  - For the years 2005-2007, I will use the years 2008-2012 as proxy.(representative of late 2000s)
  - Proxies are used because the ACS 5 year estimates started only from year 2009.
- Similarly data is not available between the years 2013-2018.
- Eventhough there is data for 2019-2023, these are 5-year estimates by ACS.
- Later, The Scope of my study will be between 2008-2012.
- To be able to match on years, I need a new column in my Wisonsin data that matches with my education data
- 1 year estimates are less reliable and not even available for counties with populaton < 65K and cannot be used.


#### Adding new column in Wisonsin data to merge with USDA Education Data

In [298]:

wis <- wis %>%
  mutate(
    edu_year_cat = case_when(
      year >= 2000 & year <= 2004 ~ "2000",
      year >= 2005 & year <= 2007 ~ "2008-12",
      year >= 2008 & year <= 2012 ~ "2008-12",  # for later merging
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(edu_year_cat)) #Removing other rows that are NA(essentially years > 2012)


#### Joining Wisonsin Data with Education data

In [299]:
wis_edu <- wis %>%
  left_join(edu_long, by = c("FIPS" = "FIPS", "edu_year_cat" = "Year"))
head(wis_edu)

Warning message in left_join(., edu_long, by = c(FIPS = "FIPS", edu_year_cat = "Year")):
"Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 233 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning."


,county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,recid_180d_violent,County_Name,FIPS,edu_year_cat,State,Area.name,2013 Urban Influence Code,2013 Rural-urban Continuum Code,Measure,Percent
,<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults who are not high school graduates,14.45220
2,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults who are high school graduates (or equivalent),29.12505
3,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults completing some college or associate degree,28.74796
4,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults with a bachelor's degree or higher,27.67479
5,30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,0,Kenosha,55059,2000,WI,Kenosha County,1,1,Percent of adults who are not high school graduates,16.50000
6,30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,0,Kenosha,55059,2000,WI,Kenosha County,1,1,Percent of adults who are high school graduates (or equivalent),33.40000


#### Checking data after merge

In [300]:
table(wis_edu$edu_year_cat, useNA = "ifany") 


   2000 2008-12 
1666740 2730560 

In [301]:
head(wis_edu)

,county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,recid_180d_violent,County_Name,FIPS,edu_year_cat,State,Area.name,2013 Urban Influence Code,2013 Rural-urban Continuum Code,Measure,Percent
,<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults who are not high school graduates,14.45220
2,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults who are high school graduates (or equivalent),29.12505
3,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults completing some college or associate degree,28.74796
4,40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,0,Milwaukee,55079,2008-12,WI,Milwaukee County,1,1,Percent of adults with a bachelor's degree or higher,27.67479
5,30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,0,Kenosha,55059,2000,WI,Kenosha County,1,1,Percent of adults who are not high school graduates,16.50000
6,30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,0,Kenosha,55059,2000,WI,Kenosha County,1,1,Percent of adults who are high school graduates (or equivalent),33.40000


#### Pivoting the table to add new columns representing all Education related data

In [302]:
wis_edu_clean <- wis_edu %>%
  filter(!is.na(Measure))
# Pivot wider using all columns except Measure and Percent
wis_edu_wide <- wis_edu_clean %>%
  pivot_wider(
    names_from = "Measure",    
    values_from = "Percent",
    values_fn = mean           # handle duplicates by taking the mean
  )

In [303]:
head(wis_edu_wide)

county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,FIPS,edu_year_cat,State,Area.name,2013 Urban Influence Code,2013 Rural-urban Continuum Code,Percent of adults who are not high school graduates,Percent of adults who are high school graduates (or equivalent),Percent of adults completing some college or associate degree,Percent of adults with a bachelor's degree or higher
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,55079,2008-12,WI,Milwaukee County,1,1,14.452197,29.12505,28.74796,27.67479
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,55059,2000,WI,Kenosha County,1,1,16.500000,33.40000,31.00000,19.20000
30,7,M,African American,1009,Misdemeanor,Resisting Officer,17,17,0,⋯,55059,2000,WI,Kenosha County,1,1,16.500000,33.40000,31.00000,19.20000
30,12,M,Caucasian,232,Criminal Traffic,Operating While Intoxicated,65,65,0,⋯,55059,2000,WI,Kenosha County,1,1,16.500000,33.40000,31.00000,19.20000
40,17,M,African American,694,Criminal Traffic,OAR/OAS,51,51,1,⋯,55079,2000,WI,Milwaukee County,1,1,19.800000,29.40000,27.20000,23.60000
67,17,M,African American,1175,Misdemeanor,Drug Possession,55,55,1,⋯,55133,2008-12,WI,Waukesha County,1,1,4.399026,25.38489,30.78018,39.43591


In [304]:
nrow(wis_edu_wide)

[1] 1093576

The merged data has the following education columns:
- 2013 Urban Influence Code	and 2013 Rural-urban Continuum Code - A code used to identify Rural and Urban areas. Between 1-12 with 1 being counties with >1 million population(metropolitan)
- Percentage of high school graduates, college graduates, higher than a college degree.

## B. USDA Unemployment data merge and cleanup with main Wisconsin Data

In [305]:
une <- read.csv("Unemployment2023.csv", header = TRUE) #Read the USDA unemployment data

In [306]:
head(une) #Checking how the data looks like

,FIPS_Code,State,Area_Name,Attribute,Value
,<int>,<chr>,<chr>,<chr>,<dbl>
1,0,US,United States,Civilian_labor_force_2000,142601576
2,0,US,United States,Employed_2000,136904853
3,0,US,United States,Unemployed_2000,5696723
4,0,US,United States,Unemployment_rate_2000,4
5,0,US,United States,Civilian_labor_force_2001,143786537
6,0,US,United States,Employed_2001,136977996


In [307]:
une_clean <- une %>%
  filter(FIPS_Code >= 55001 & FIPS_Code <= 55141)  %>% #Filter FIPS codes corresponding to Wisconsin
  # Extract year from Attribute column for the merge
  mutate(year = as.integer(str_extract(Attribute, "\\d{4}"))) %>%
  rename(FIPS = FIPS_Code)

In [308]:
# Filter the desired attributes - mainly the Codes that represent Rural/Urban
une_filtered <- une_clean %>%
  filter(Attribute %in% c(
    "Rural_Urban_Continuum_Code_2023",
    "Urban_Influence_Code_2013",
    "Metro_2023"
  ))

In [309]:
# Pivot to get one FIPS per row to join with wisconsin dataset that has education data
une_wide <- une_filtered %>%
  select(FIPS, Attribute, Value) %>%
  pivot_wider(names_from = Attribute, values_from = Value)
#une_wide now has basic columns(not specific to any year) in a wide format

wis_edu_une_temp <- wis_edu_wide %>% #Joining wisonsin_education data with unemployment data 
#with basic columns(Rural/urban codes)
  left_join(une_wide, by = "FIPS") #Since there is no year for the basic columns, join by FIPS alone
#Now this temporary joined file has previous Wisconsin+Education data plus the basic columns from unemployment data

In [310]:
#Filter other columns except the basic ones to further join
une_filtered_2 <- une_clean %>% 
  filter(!Attribute %in% c(
    "Rural_Urban_Continuum_Code_2023",
    "Urban_Influence_Code_2013",
    "Metro_2023"
  ))

une_filtered_2 <- une_filtered_2 %>%
  mutate(
    Attribute = str_remove(Attribute, "_\\d{4}$") ) # cleanup '_year' in attribute column
head(une_filtered_2) #Getting the other data with year attached to it as a column

,FIPS,State,Area_Name,Attribute,Value,year
,<int>,<chr>,<chr>,<chr>,<dbl>,<int>
1,55001,WI,"Adams County, WI",Civilian_labor_force,8892.0,2000
2,55001,WI,"Adams County, WI",Employed,8502.0,2000
3,55001,WI,"Adams County, WI",Unemployed,390.0,2000
4,55001,WI,"Adams County, WI",Unemployment_rate,4.4,2000
5,55001,WI,"Adams County, WI",Civilian_labor_force,9093.0,2001
6,55001,WI,"Adams County, WI",Employed,8632.0,2001


In [311]:
# Merge by FIPS and Year
wis_edu_une_merged <- wis_edu_une_temp %>%
  left_join(une_filtered_2, by = c("FIPS", "year"))

head(wis_edu_une_merged)

Warning message in left_join(., une_filtered_2, by = c("FIPS", "year")):
"Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 2843 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning."


county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,Percent of adults who are high school graduates (or equivalent),Percent of adults completing some college or associate degree,Percent of adults with a bachelor's degree or higher,Rural_Urban_Continuum_Code_2023,Urban_Influence_Code_2013,Metro_2023,State.y,Area_Name,Attribute,Value
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,29.12505,28.74796,27.67479,1,1,1,WI,"Milwaukee County, WI",Civilian_labor_force,475758.0
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,29.12505,28.74796,27.67479,1,1,1,WI,"Milwaukee County, WI",Employed,435411.0
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,29.12505,28.74796,27.67479,1,1,1,WI,"Milwaukee County, WI",Unemployed,40347.0
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,29.12505,28.74796,27.67479,1,1,1,WI,"Milwaukee County, WI",Unemployment_rate,8.5
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,33.40000,31.00000,19.20000,3,1,1,WI,"Kenosha County, WI",Civilian_labor_force,80882.0
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,33.40000,31.00000,19.20000,3,1,1,WI,"Kenosha County, WI",Employed,77651.0


In [312]:
wis_edu_une_merged <- wis_edu_une_merged %>%
  filter(year >= 2000 & year <= 2012) #For now filterung years 2000-2012
# Will restrict the analysis to 2008-2012 later on

In [313]:
# Pivot wider to get each attribute as a column
wis_edu_une_wide <- wis_edu_une_merged %>%
  pivot_wider(
    names_from = Attribute,
    values_from = Value
  )
head(wis_edu_une_wide)

county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,Percent of adults with a bachelor's degree or higher,Rural_Urban_Continuum_Code_2023,Urban_Influence_Code_2013,Metro_2023,State.y,Area_Name,Civilian_labor_force,Employed,Unemployed,Unemployment_rate
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,27.67479,1,1,1,WI,"Milwaukee County, WI",475758,435411,40347,8.5
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,19.20000,3,1,1,WI,"Kenosha County, WI",80882,77651,3231,4.0
30,7,M,African American,1009,Misdemeanor,Resisting Officer,17,17,0,⋯,19.20000,3,1,1,WI,"Kenosha County, WI",80882,77651,3231,4.0
30,12,M,Caucasian,232,Criminal Traffic,Operating While Intoxicated,65,65,0,⋯,19.20000,3,1,1,WI,"Kenosha County, WI",80882,77651,3231,4.0
40,17,M,African American,694,Criminal Traffic,OAR/OAS,51,51,1,⋯,23.60000,1,1,1,WI,"Milwaukee County, WI",473934,448487,25447,5.4
67,17,M,African American,1175,Misdemeanor,Drug Possession,55,55,1,⋯,39.43591,1,1,1,WI,"Waukesha County, WI",212463,204297,8166,3.8


#### Checking some redundant columns

In [314]:
# Check if values of Urban Influence Code are exactly the same in education and unemployment data
all_equal_UI_code <- all(wis_edu_une_wide$`2013 Urban Influence Code` == wis_edu_une_wide$Urban_Influence_Code_2013, na.rm = TRUE)
all_equal_UI_code


[1] TRUE

- Since the Urban Influence Code matches between my education data and unemployment data from USDA, I am going to get rid of one of these.
- Rural_Urban_Continuum_Code_2023 is not in my scope and I am going to use 2013 Rural-urban Continuum Code instead. 'State.y''Area_Name' are also redundant.

In [315]:
# Removing redundant and unncessary columns
wis_edu_une_wide <- wis_edu_une_wide %>%
  select(
    -c(
      Rural_Urban_Continuum_Code_2023,
      Urban_Influence_Code_2013,
      State.y,
      Area_Name
    )
  )

In [316]:
head(wis_edu_une_wide)

county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,2013 Rural-urban Continuum Code,Percent of adults who are not high school graduates,Percent of adults who are high school graduates (or equivalent),Percent of adults completing some college or associate degree,Percent of adults with a bachelor's degree or higher,Metro_2023,Civilian_labor_force,Employed,Unemployed,Unemployment_rate
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,1,14.452197,29.12505,28.74796,27.67479,1,475758,435411,40347,8.5
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,1,16.500000,33.40000,31.00000,19.20000,1,80882,77651,3231,4.0
30,7,M,African American,1009,Misdemeanor,Resisting Officer,17,17,0,⋯,1,16.500000,33.40000,31.00000,19.20000,1,80882,77651,3231,4.0
30,12,M,Caucasian,232,Criminal Traffic,Operating While Intoxicated,65,65,0,⋯,1,16.500000,33.40000,31.00000,19.20000,1,80882,77651,3231,4.0
40,17,M,African American,694,Criminal Traffic,OAR/OAS,51,51,1,⋯,1,19.800000,29.40000,27.20000,23.60000,1,473934,448487,25447,5.4
67,17,M,African American,1175,Misdemeanor,Drug Possession,55,55,1,⋯,1,4.399026,25.38489,30.78018,39.43591,1,212463,204297,8166,3.8


## C. Census Poverty data merge and cleanup with main Wisconsin Data

In [317]:
#Data Source : https://www.census.gov/programs-surveys/saipe/data/api.html

base_url <- "https://api.census.gov/data/timeseries/poverty/saipe"
years <- 2000:2018

all_pov <- list()

for (yr in years) {
  params <- list(
    get = "NAME,SAEPOVALL_PT,SAEPOVRTALL_PT",  #estimated percentage of all people (all ages) living below the poverty level
    `for` = "county:*",
    `in` = "state:55",   # Wisconsin
    time = yr
  )
  
  resp <- GET(base_url, query = params)
  tmp <- fromJSON(content(resp, "text", encoding = "UTF-8"))
  
  colnames(tmp) <- tmp[1, ]
  tmp <- tmp[-1, ] %>% as.data.frame(stringsAsFactors = FALSE)
  tmp$year <- yr
  all_pov[[as.character(yr)]] <- tmp
}

df_pov <- bind_rows(all_pov) %>%
  mutate(
    SAEPOVALL_PT = as.numeric(SAEPOVALL_PT),
    SAEPOVRTALL_PT = as.numeric(SAEPOVRTALL_PT),
    county_fips = paste0(state, county)
  ) %>%
  select(year, county_fips, NAME, SAEPOVALL_PT, SAEPOVRTALL_PT)
# Save df_saipe to a CSV file
write.csv(df_pov, file = "Wisconsin_county_poverty_2000_2018.csv", 
          row.names = FALSE)

#SAEPOVALL_PT (point estimate only)
#SAEPOVRTALL_PT (point estimate with margin of error) - Percentage

In [318]:
pov <- read.csv("Wisconsin_county_poverty_2000_2018.csv", header = TRUE) #Reading the Poverty data
pov <- pov %>%
  rename(FIPS = county_fips)
head(pov)

,year,FIPS,NAME,SAEPOVALL_PT,SAEPOVRTALL_PT
,<int>,<int>,<chr>,<int>,<dbl>
1,2000,55001,Adams County,2315,12.3
2,2000,55003,Ashland County,2015,12.5
3,2000,55005,Barron County,4168,9.3
4,2000,55007,Bayfield County,1718,11.4
5,2000,55009,Brown County,14835,6.6
6,2000,55011,Buffalo County,1166,8.5


In [319]:

# Ensure the columns have the same type
wis_edu_une_wide <- wis_edu_une_wide %>%
  mutate(FIPS = as.integer(FIPS),
         year = as.integer(year))

pov <- pov %>%
  mutate(FIPS = as.integer(FIPS),
         year = as.integer(year))

# Merge the datasets by FIPS and year
wis_edu_une_pov <- wis_edu_une_wide %>%
  left_join(pov, by = c("FIPS", "year"))

# Check result
head(wis_edu_une_pov)


county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,⋯,Percent of adults completing some college or associate degree,Percent of adults with a bachelor's degree or higher,Metro_2023,Civilian_labor_force,Employed,Unemployed,Unemployment_rate,NAME,SAEPOVALL_PT,SAEPOVRTALL_PT
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<dbl>
40,3,M,African American,436,Felony,Drug Possession,40,41,0,⋯,28.74796,27.67479,1,475758,435411,40347,8.5,Milwaukee County,209201,22.5
30,6,M,Caucasian,1049,Misdemeanor,Disorderly Conduct,39,40,0,⋯,31.00000,19.20000,1,80882,77651,3231,4.0,Kenosha County,12262,8.2
30,7,M,African American,1009,Misdemeanor,Resisting Officer,17,17,0,⋯,31.00000,19.20000,1,80882,77651,3231,4.0,Kenosha County,12262,8.2
30,12,M,Caucasian,232,Criminal Traffic,Operating While Intoxicated,65,65,0,⋯,31.00000,19.20000,1,80882,77651,3231,4.0,Kenosha County,12262,8.2
40,17,M,African American,694,Criminal Traffic,OAR/OAS,51,51,1,⋯,27.20000,23.60000,1,473934,448487,25447,5.4,Milwaukee County,123305,13.5
67,17,M,African American,1175,Misdemeanor,Drug Possession,55,55,1,⋯,30.78018,39.43591,1,212463,204297,8166,3.8,Waukesha County,14014,3.8


In [320]:
#Remove redundant column
wis_edu_une_pov <- wis_edu_une_pov %>%
  select(-NAME)

In [321]:
colnames(wis_edu_une_pov)

[1] "county"                                                         
 [2] "new_id"                                                         
 [3] "sex"                                                            
 [4] "race"                                                           
 [5] "judge_id"                                                       
 [6] "case_type"                                                      
 [7] "wcisclass"                                                      
 [8] "age_offense"                                                    
 [9] "age_judge"                                                      
[10] "prior_felony"                                                   
[11] "prior_misdemeanor"                                              
[12] "prior_criminal_traffic"                                         
[13] "highest_severity"                                               
[14] "release"                                                        
[15] "probation"                                                      
[16] "is_recid_new"                                                   
[17] "recid_180d"                                                     
[18] "zip"                                                            
[19] "pct_black"                                                      
[20] "pct_hisp"                                                       
[21] "pct_male"                                                       
[22] "pct_rural"                                                      
[23] "pct_urban"                                                      
[24] "pct_college"                                                    
[25] "pct_food_stamps"                                                
[26] "pop_dens"                                                       
[27] "pct_somecollege"                                                
[28] "med_hhinc"                                                      
[29] "year"                                                           
[30] "jail"                                                           
[31] "train_test_split_caselevel"                                     
[32] "train_test_split_deflevel"                                      
[33] "prior_charges_severity7"                                        
[34] "prior_charges_severity8"                                        
[35] "prior_charges_severity9"                                        
[36] "prior_charges_severity10"                                       
[37] "prior_charges_severity11"                                       
[38] "prior_charges_severity12"                                       
[39] "prior_charges_severity13"                                       
[40] "prior_charges_severity14"                                       
[41] "prior_charges_severity15"                                       
[42] "prior_charges_severity16"                                       
[43] "prior_charges_severity17"                                       
[44] "prior_charges_severity18"                                       
[45] "prior_charges_severity19"                                       
[46] "prior_charges_severity20"                                       
[47] "prior_charges_severity21"                                       
[48] "max_hist_jail"                                                  
[49] "min_hist_jail"                                                  
[50] "avg_hist_jail"                                                  
[51] "median_hist_jail"                                               
[52] "all_races"                                                      
[53] "violent_crime"                                                  
[54] "recid_180d_violent"                                             
[55] "County_Name"                                                    
[56] "FIPS"                                                           
[57] "edu_year_cat"      

- Above is a combined dataset that contains the Main Wisconsin data, Education and Unemployment data from USDA and Poverty data from Census(through API).

## D. Final Cleanup before EDA

#### Getting rid of redundant and unnecessary columns

In [327]:
wis_edu_une_filtered <- wis_edu_une_pov %>%
  select(-State.x, -Area.name, -Metro_2023) #zip is psuedo. Keeping it as education data differs by this.


#### Defining the scope of the Analysis interms of years and dropping NA rows(No recidivism indicator)

In [328]:
# Scope of the analysis is 2008-2012
wis_edu_une_filtered <- wis_edu_une_filtered %>%
filter(year >= 2008 & year <= 2012)

#Missing recidivism indicator year-wise percentage
wis_edu_une_filtered %>%
  group_by(year) %>%
  summarize(missing_recid = sum(is.na(is_recid_new)),
            total = n(),
            pct_missing = 100 * missing_recid / total)


year,missing_recid,total,pct_missing
<int>,<int>,<int>,<dbl>
2008,3461,89979,3.846453
2009,3486,83823,4.158763
2010,3527,76949,4.583555
2011,3575,73651,4.853973
2012,3893,74845,5.201416


In [329]:
# Remove rows that have missing values in is_recid_new
wis_edu_une_filtered <- wis_edu_une_filtered %>% drop_na(is_recid_new)
nrow(wis_edu_une_filtered)

[1] 381305

In [330]:
# Missing value counts ordered from most to least
colSums(is.na(wis_edu_une_filtered)) %>% sort(decreasing = TRUE)

jail 
                                                         227583 
                                                      probation 
                                                         120138 
                                                  max_hist_jail 
                                                          70491 
                                                  min_hist_jail 
                                                          70491 
                                                  avg_hist_jail 
                                                          70491 
                                               median_hist_jail 
                                                          70491 
                                                     recid_180d 
                                                          13683 
                                             recid_180d_violent 
                                                          13683 
                                                            zip 
                                                           7384 
                                                         county 
                                                              0 
                                                         new_id 
                                                              0 
                                                            sex 
                                                              0 
                                                           race 
                                                              0 
                                                       judge_id 
                                                              0 
                                                      case_type 
                                                              0 
                                                      wcisclass 
                                                              0 
                                                    age_offense 
                                                              0 
                                                      age_judge 
                                                              0 
                                                   prior_felony 
                                                              0 
                                              prior_misdemeanor 
                                                              0 
                                         prior_criminal_traffic 
                                                              0 
                                               highest_severity 
                                                              0 
                                                        release 
                                                              0 
                                                   is_recid_new 
                                                              0 
                                                      pct_black 
                                                              0 
                                                       pct_hisp 
                                                              0 
                                                       pct_male 
                                                              0 
                                                      pct_rural 
                                                              0 
                                                      pct_urban 
                                                              0 
                                                    pct_college 
                                                              0 
                                                pct_food_stamps 
                                                              0 
                             

In [331]:
# Which rows have jail data missing?
jail_missing <- wis_edu_une_filtered %>% 
  filter(is.na(jail)) %>%
select(county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,jail,probation)

jail_missing

county,new_id,sex,race,judge_id,case_type,wcisclass,age_offense,age_judge,prior_felony,jail,probation
<int>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<int>
45,55,F,African American,1493,Misdemeanor,Drug Possession,19,20,0,NA,1
37,62,F,African American,1557,Misdemeanor,Battery,19,19,0,NA,1
64,78,F,African American,251,Misdemeanor,Theft,17,18,0,NA,1
7,98,F,Caucasian,816,Felony,Other Homicide,40,41,0,NA,1
53,106,F,Caucasian,522,Criminal Traffic,Other Misdemeanor,18,19,0,NA,NA
67,116,M,Caucasian,540,Felony,Other Crimes Against Children,27,32,0,NA,1
70,118,M,Caucasian,64,Misdemeanor,Disorderly Conduct,24,25,0,NA,NA
32,120,M,Caucasian,1178,Criminal Traffic,OAR/OAS,19,20,0,NA,NA
71,126,M,American Indian or Alaskan Native,1489,Felony,Other Bodily Security,25,25,1,NA,NA


In [332]:
# Percentage of jail data missing for those on probation case_type wise
wis_edu_une_filtered %>%
  filter(is.na(jail) & probation == 1) %>%
  count(case_type) %>%
  mutate(prop = n / sum(n))

case_type,n,prop
<chr>,<int>,<dbl>
Criminal Traffic,4664,0.03742157
Felony,64831,0.52017106
Misdemeanor,55139,0.44240737


In [333]:
#### Impute probation NA as 0
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    probation = ifelse(is.na(probation), 0, probation)
  )
#### Impute Jail as 0 only for those that have Probation =1 and not Felony cases
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    jail = ifelse(is.na(jail) & probation == 1 &
                    case_type != "Felony", 0, jail)
  )

In [334]:
colSums(is.na(wis_edu_une_filtered)) %>% sort(decreasing = TRUE)

jail 
                                                         167780 
                                                  max_hist_jail 
                                                          70491 
                                                  min_hist_jail 
                                                          70491 
                                                  avg_hist_jail 
                                                          70491 
                                               median_hist_jail 
                                                          70491 
                                                     recid_180d 
                                                          13683 
                                             recid_180d_violent 
                                                          13683 
                                                            zip 
                                                           7384 
                                                         county 
                                                              0 
                                                         new_id 
                                                              0 
                                                            sex 
                                                              0 
                                                           race 
                                                              0 
                                                       judge_id 
                                                              0 
                                                      case_type 
                                                              0 
                                                      wcisclass 
                                                              0 
                                                    age_offense 
                                                              0 
                                                      age_judge 
                                                              0 
                                                   prior_felony 
                                                              0 
                                              prior_misdemeanor 
                                                              0 
                                         prior_criminal_traffic 
                                                              0 
                                               highest_severity 
                                                              0 
                                                        release 
                                                              0 
                                                      probation 
                                                              0 
                                                   is_recid_new 
                                                              0 
                                                      pct_black 
                                                              0 
                                                       pct_hisp 
                                                              0 
                                                       pct_male 
                                                              0 
                                                      pct_rural 
                                                              0 
                                                      pct_urban 
                                                              0 
                                                    pct_college 
                                                              0 
                                                pct_food_stamps 
                                                              0 
                             

- My focus will not be on the variables that have the most NAs above. Hence, I will be leaving it as it is.

## E. Feature Engineering

In [335]:
# Add some flags
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    # Combine all prior offenses into a single column
    total_priors = prior_felony + prior_misdemeanor + prior_criminal_traffic,

    # Binary indicator: any prior record
    has_priors = if_else(total_priors > 0, 1, 0),

    # Binary indicator: any jail history
    has_jail_history = if_else(max_hist_jail > 0 | avg_hist_jail > 0, 1, 0),
  )

# Bins for ages

wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    # Age groups at time of offense
    age_group = case_when(
      age_offense < 25 ~ "Under_25",
      age_offense < 35 ~ "25_to_34",
      age_offense < 50 ~ "35_to_49",
      TRUE ~ "50_plus"
    ),
    
    # Difference between age at offense and age at judgement
    case_duration = age_judge - age_offense
  )

# Sum all "prior_charges_severity" columns into a single index
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    total_prior_severity = rowSums(select(., starts_with("prior_charges_severity")), na.rm = TRUE)
  )


In [336]:
# total_prior_severity has a total of all the severity charges. Dropping individual prior_charges_severity columns
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  select(-starts_with("prior_charges_severity"))

In [337]:
colnames(wis_edu_une_filtered)

[1] "county"                                                         
 [2] "new_id"                                                         
 [3] "sex"                                                            
 [4] "race"                                                           
 [5] "judge_id"                                                       
 [6] "case_type"                                                      
 [7] "wcisclass"                                                      
 [8] "age_offense"                                                    
 [9] "age_judge"                                                      
[10] "prior_felony"                                                   
[11] "prior_misdemeanor"                                              
[12] "prior_criminal_traffic"                                         
[13] "highest_severity"                                               
[14] "release"                                                        
[15] "probation"                                                      
[16] "is_recid_new"                                                   
[17] "recid_180d"                                                     
[18] "zip"                                                            
[19] "pct_black"                                                      
[20] "pct_hisp"                                                       
[21] "pct_male"                                                       
[22] "pct_rural"                                                      
[23] "pct_urban"                                                      
[24] "pct_college"                                                    
[25] "pct_food_stamps"                                                
[26] "pop_dens"                                                       
[27] "pct_somecollege"                                                
[28] "med_hhinc"                                                      
[29] "year"                                                           
[30] "jail"                                                           
[31] "train_test_split_caselevel"                                     
[32] "train_test_split_deflevel"                                      
[33] "max_hist_jail"                                                  
[34] "min_hist_jail"                                                  
[35] "avg_hist_jail"                                                  
[36] "median_hist_jail"                                               
[37] "all_races"                                                      
[38] "violent_crime"                                                  
[39] "recid_180d_violent"                                             
[40] "County_Name"                                                    
[41] "FIPS"                                                           
[42] "edu_year_cat"                                                   
[43] "2013 Urban Influence Code"                                      
[44] "2013 Rural-urban Continuum Code"                                
[45] "Percent of adults who are not high school graduates"            
[46] "Percent of adults who are high school graduates (or equivalent)"
[47] "Percent of adults completing some college or associate degree"  
[48] "Percent of adults with a bachelor's degree or higher"           
[49] "Civilian_labor_force"                                           
[50] "Employed"                                                       
[51] "Unemployed"                                                     
[52] "Unemployment_rate"                                              
[53] "SAEPOVALL_PT"                                                   
[54] "SAEPOVRTALL_PT"                                                 
[55] "total_priors"                                                   
[56] "has_priors"                                                     
[57] "has_jail_history"  

In [338]:
# Adding a weighted index for education data to have one column representing education data to prevent multicollinearity
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    edu_index = 
      1 * `Percent of adults who are not high school graduates` +
      2 * `Percent of adults who are high school graduates (or equivalent)` +
      3 * `Percent of adults completing some college or associate degree` +
      4 * `Percent of adults with a bachelor\'s degree or higher`
  )


In [339]:
# Using pct_college and some_college data and aggregating that to get county-wise data as these are zip-wise

# Aggregate ZIP-level education to county-level
county_edu_summary <- wis_edu_une_filtered %>%
  group_by(County_Name) %>%
  summarise(
    avg_pct_college = mean(pct_college, na.rm = TRUE),
    avg_pct_somecollege = mean(pct_somecollege, na.rm = TRUE)
  )

# Merge back to the main dataset
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  left_join(county_edu_summary,by = c("County_Name"))

In [340]:
# Using pct_college and some_college data and aggregating that to get county-wise and year-wise

# Aggregate ZIP-level education to county-level and year averages
county_year_edu_summary <- wis_edu_une_filtered %>%
  group_by(County_Name,year) %>%
  summarise(
    avg_year_pct_college = mean(pct_college, na.rm = TRUE),
    avg_year_pct_somecollege = mean(pct_somecollege, na.rm = TRUE)
  )

# Merge back to the main dataset
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  left_join(county_year_edu_summary,by = c("County_Name", "year"))

# Adding a weighted index for education data to have one column representing education data to prevent multicollinearity
# For the base education data in the Wisconsin dataset
# Will decide which index to use during modelling
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    edu_index2 = 
      1 * avg_year_pct_somecollege +
      2 * avg_year_pct_college     
  )


`summarise()` has grouped output by 'County_Name'. You can override using the
`.groups` argument.


In [341]:
# A combined total priors are present in total_priors field
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  select(-prior_felony, -prior_misdemeanor, -prior_criminal_traffic)
glimpse(wis_edu_une_filtered)

Rows: 381,305
Columns: 63
$ county                                                            <int> 40, …
$ new_id                                                            <int> 3, 2…
$ sex                                                               <chr> "M",…
$ race                                                              <chr> "Afr…
$ judge_id                                                          <int> 436,…
$ case_type                                                         <chr> "Fel…
$ wcisclass                                                         <chr> "Dru…
$ age_offense                                                       <int> 40, …
$ age_judge                                                         <int> 41, …
$ highest_severity                                                  <int> 7, 7…
$ release                                                           <int> 1, 0…
$ probation                                                         <dbl> 0, 0…
$ is_recid_new

In [342]:
#Impute NA with 0 for has_jail_history (Planning to use this in modelling)
wis_edu_une_filtered <- wis_edu_une_filtered %>%
  mutate(
    has_jail_history = ifelse(is.na(has_jail_history), 0, has_jail_history)
  )

In [343]:
colSums(is.na(wis_edu_une_filtered)) %>% sort(decreasing = TRUE)

jail 
                                                         167780 
                                                  max_hist_jail 
                                                          70491 
                                                  min_hist_jail 
                                                          70491 
                                                  avg_hist_jail 
                                                          70491 
                                               median_hist_jail 
                                                          70491 
                                                     recid_180d 
                                                          13683 
                                             recid_180d_violent 
                                                          13683 
                                                            zip 
                                                           7384 
                                                         county 
                                                              0 
                                                         new_id 
                                                              0 
                                                            sex 
                                                              0 
                                                           race 
                                                              0 
                                                       judge_id 
                                                              0 
                                                      case_type 
                                                              0 
                                                      wcisclass 
                                                              0 
                                                    age_offense 
                                                              0 
                                                      age_judge 
                                                              0 
                                               highest_severity 
                                                              0 
                                                        release 
                                                              0 
                                                      probation 
                                                              0 
                                                   is_recid_new 
                                                              0 
                                                      pct_black 
                                                              0 
                                                       pct_hisp 
                                                              0 
                                                       pct_male 
                                                              0 
                                                      pct_rural 
                                                              0 
                                                      pct_urban 
                                                              0 
                                                    pct_college 
                                                              0 
                                                pct_food_stamps 
                                                              0 
                                                       pop_dens 
                                                              0 
                                                pct_somecollege 
                                                              0 
                                                      med_hhinc 
                                                              0 
                             

In [344]:
nrow(wis_edu_une_filtered)

[1] 381305

#### Write data for EDA and Modelling

In [345]:
write.csv(wis_edu_une_filtered, "wis_edu_une_pov.csv", row.names = FALSE)